# 01 EDA: motor thermal dataset

Sampling is 2 Hz. `torque` and `motor_speed` are **estimates** derived from currents and voltages, not sensor readings. All computation lives in `motor.eda`; cells only call it and plot.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from motor.eda import peak_distribution, profile_summary

ROOT = Path.cwd().resolve()
while not (ROOT / 'data').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
df = pd.read_parquet(ROOT / 'data' / 'processed' / 'measures.parquet')
summary = profile_summary(df)
print('profiles:', len(summary))

## Profile durations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(summary['profile_id'].astype(str), summary['duration_h'])
ax.set_xlabel('profile_id')
ax.set_ylabel('duration [h]')
ax.tick_params(axis='x', labelrotation=90, labelsize=6)
plt.show()

## Operating region (estimated speed x estimated torque, colored by coolant)

In [ ]:
sample = df.iloc[::50]
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(sample['motor_speed'], sample['torque'], c=sample['coolant'], s=2)
fig.colorbar(sc, label='coolant [degC]')
ax.set_xlabel('motor_speed [rpm] (estimate)')
ax.set_ylabel('torque [N m] (estimate)')
plt.show()

## Per-window peak distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex='row')
for r, target in enumerate(['pm', 'stator_winding']):
    for c, minutes in enumerate([10, 30, 60]):
        peaks = peak_distribution(df, target, window_s=minutes * 60)
        axes[r, c].hist(peaks, bins=40)
        axes[r, c].set_title(f'{target}, {minutes} min windows (n={len(peaks)})')
        axes[r, c].set_xlabel('peak [degC]')
plt.tight_layout()
plt.show()